# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**, identified by `order_id`.
- Aggregate `order_items`, `order_payments`, and `order_reviews` to `order_id` before joining them to `orders`.
- Join `products`, category translations, and sellers to item-level data before aggregating those fields to order level.
- `customer_id` identifies the customer record attached to one order; `customer_unique_id` may repeat because one person can place multiple orders.
- Check row counts and duplicate `order_id` values after every major merge.
- Define targets only where the outcome is observable. An undelivered order is not automatically an on-time order, and a missing review is not a low review.
- State when a prediction would be made, then exclude fields that would not be available at that time.


## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



items: 28,343 rows x 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87
1,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.9,12.69
2,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.9,11.85


... (8 outputs left out)


geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [5]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


,table,expected_key,duplicate_keys,missing_key_values
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,order_items,order_id + order_item_id,0,0
3,order_payments,order_id + payment_sequential,0,0
4,order_reviews,review record (review_id may repeat),0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,geolocation_zip_prefix,geolocation_zip_code_prefix,0,0
8,translations,product_category_name,0,0


Item rows sharing an order_id: 3,552
Payment rows sharing an order_id: 1,154
Review rows sharing an order_id: 137
Repeated customer_unique_id values: 236


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
What order, product, or customer level factors are associated with order fulfillment problems for Olist's marketplace, such as late delivery and cancelations. The goal is to find out whether these factors can be used to catch these problems early on to help Olist prevent or mitigate risks in the fulfillment process.

- Why this question matters:
Fulfillment problems affect customer trust and marketplace reputation. If Olist's operations team can identify order-level risk signals near the time of purchase, they can prioritize intervention, such as confirming stock with a seller, expediting shipment, or proactively contacting the customer, before an order fails to meet expectations.

- Tables/files used:
This project primarily uses data from orders, order_items, products, and customers. sellers and order_reviews will likely be incorporated as profiling and cleaning continue for this project; geolocation_zip_prefix may not be needed depending on final scope of the problem in question.

- Row meaning of each important table:
    * orders: one row per order (25,000 rows); the base table for this analysis
    * customers: one row per order-specific customer record (25,000 rows); customer_unique_id can repeat across orders for repeat buyers
    * order_items: one row per item line within an order (28,343 rows); orders with multiple items have multiple rows and must be aggregated to order_id
    * order_payments: one row per payment transaction within an order (26,154 rows); orders with split or installment payments have multiple rows and must be aggregated to order_id
    * order_reviews: one row per review; review_id is not guaranteed unique and 137 orders have multiple review rows; not every order has a review (24,934 review rows vs. 25,000 orders)
    * products: one row per product (13,344 rows)
    * sellers: one row per seller (2,234 rows)

- Early data quality issues:
Profiling showed several data quality issues to address before the analysis. The orders table has missing values concentrated in delivery-related timestamps (39 missing order_approved_at, 452 missing order_delivered_carrier_date, 775 missing order_delivered_customer_date), consistent with orders that were canceled or an order that never completed delivery. The products table has 230 rows missing category, name, description, and photo count together, representing incomplete listings, plus one additional row also missing all physical attributes. Two product categories present in products ("pc_gamer" and "portateis_cozinha_e_preparadores_de_alimentos") have no matching entry in the translations table. Zip code prefix length varies inconsistently across tables and will need standardized formatting before being used in a location-based join. Finally, cancellation is a rare outcome (0.6% of orders), creating a class imbalance that will need to be addressed in later modeling. Wether a location-based join is necessary is still unknown. 

- Initial cleaning plan:
Order_items and order_payments will be aggregated to one row per order_id (For total item count and price for items, total payment value and primary payment type for payments) before joining to the orders base table. Order_reviews will also need to be aggregated or deduplicated to order_id, with a way to treat orders with no review rather than assuming a missing review will equal a low score. Delivery timestamp fields will need validation so that a "late delivery" flag is calculated only for orders that were actually delivered. The two untranslated product categories will be manually mapped or excluded from English-language reporting. Customer_zip_code_prefix contains a mix of 4-digit and 5-digit values, which was confirmed to reflect legitimate variation in the source data rather than a data quality error. This is not itself a problem, but it could cause join mismatches if zip prefix values are formatted differently across customers, sellers, and geolocation tables. Whether this needs to be addressed depends on whether a location-based join is pursued in a later milestone. Class imbalance in the cancellation target will be addressed through techniques such as resampling, class weighting, or shifting toward the broader "problem order" definition.

- Possible target/outcome:
I am considering two related binary target variables. The first is a target focused on order cancellation, predicting whether a given order will be cancelled. The second is a more broad "problem order" indicator that combines cancellation and late delivery (actual delivery date later than estimated delivery date) into a single variable. Review score may also be used as an additional signal of order problems as the project progresses. Which target will be used depends on further profiling and analysis. 

- Ethics, privacy, or governance concerns:
This dataset includes customer and seller identifiers along with zip code-level location data, which could act as quasi-identifiers if combined with other information. customer_id and customer_unique_id should be treated as sensitive and excluded or restricted in any shared or broadly distributed reports, since they can be linked to personal financial information such as payment amounts. seller_id carries a lower privacy risk since sellers are public business entities rather than private individuals, but seller-level findings like being flagged a high-risk for cancellations should still be handled carefully given potential reputation impact. Any location-based analysis should be aggregated to the city or state level rather than shared at the individual zip-prefix level to reduce identification risk.



### Data Profiling and exploratory data analysis

##### Data Types

In [6]:
# Checking data types for key variables for each table by reusing loop for basic data pofile.
for name, df in tables.items():
    print(f"\n{name}:")
    print(df.dtypes)



orders:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

customers:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

items:
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

payments:
order_id                 object
payment_sequential        int64
payment_type             obje

#### Data Types Observations:
After checking each variable and it's type I have come to the conclusion that data types are correctly identifed for each table.


### Missing Values

In [7]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.

# Reuse the same loop structure to check for missing values in each table.
for name, df in tables.items():
    print(f'\n{name}:')
    print(df.isna().sum())


orders:
order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 39
order_delivered_carrier_date     452
order_delivered_customer_date    775
order_estimated_delivery_date      0
dtype: int64

customers:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

items:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

payments:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

reviews:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       22008
review_comment_

In [8]:

# Check if products missing product_category_name are also missing other key product attributes
missing_category_rows = products[products["product_category_name"].isna()]
missing_all_four = missing_category_rows[
    missing_category_rows["product_name_lenght"].isna() &
    missing_category_rows["product_description_lenght"].isna() &
    missing_category_rows["product_photos_qty"].isna()
]
print(missing_all_four.shape[0])

230


In [9]:
# Verifiying the same 230 rows are all mising the same product attributes
print(products["product_category_name"].isna().sum()) 
print(missing_all_four.shape[0])                           

230
230


##### Missing Value Observations:
* orders- Starting with the top of the fulfillment pipeline probably the stage oders are approved order_approved_at is missing 39 observations. The next stage order_delivered_carrier_date, which is the date that the seller hands off an order to the carrier is missing 452 observations. The last variable that is missing observations is order_delivered_customer_date, the date that an order is delivered to the customer is missing 775 observations. This suggest that the later the stage in the fulfillment process the more missing observations there are.

* reviews- The next table with missing observations is Reviews. Review title is missing 22,008 out of the 24,932 total rows. This variable is missing about 88.3% of total rows. The actual content of the comment review_comment_message is also missing a large amount of observations. The missing count of this variable is 14,610 rows which is about 58.6% of total rows. To me this is expected, I a often guilty of just putting a review without context. My guess is negative reviews have more content than positive reviews becasue people are more likely to complain then give praise in my experience. 

* products- This is the last table in the Olist tables that has missing values. Something interesting I found is that product_category_name, product_name_lenght, product_description_lenght, and product_photos_qty all are missing the same 230 rows. It looks like something systematic suggesting the same rows in the data are missing all 4 rows. To investigate this, I stored all rows missing product_category_name and then tested how many fit the criteria of missing all other attributes as well. Afterward, I compared the total missing values in product_category_name with the set that satisfied the 4 missing attribute criteria and found that the same rows were missing in the same rows. After thinking about it this is expected. If a product name is missing then we cannot have a name length, description, or pictures. It doesn't exist. product_weight_g, product_length_cm, product_height_cm, and product_width_cm are all missing 1 value.

* customers, items, payments, sellers, geolocation_zip_prefix, and translations do not have any missing values.

#### Descriptive Statistics

##### items table

In [10]:
# Descriptive stats for quanititative variables in the items table.
items[["price", "freight_value"]].describe()

,price,freight_value
count,28343.000000,28343.000000
mean,120.529931,19.973035
std,178.492390,15.387608
min,2.200000,0.000000
25%,39.900000,13.125000
50%,75.000000,16.320000
75%,135.000000,21.235000
max,6735.000000,339.590000


In [11]:
# Check the number of rows in the items table where freight_value is equal to 0
print((items["freight_value"] == 0).sum())

108


##### payments table

In [12]:
# Descriptive stats for quanititative variables in the payments table.
payments[["payment_value", "payment_installments"]].describe()

,payment_value,payment_installments
count,26154.000000,26154.000000
mean,154.073541,2.873863
std,208.396268,2.702997
min,0.000000,1.000000
25%,56.860000,1.000000
50%,100.015000,1.000000
75%,173.247500,4.000000
max,6929.310000,24.000000


In [13]:
# Check for payments that are equal to 0.
print((payments["payment_value"] == 0).sum())

# Check what payment types are associated with payments that are equal to 0.
print(payments[payments["payment_value"] == 0]["payment_type"].value_counts())


2
payment_type
voucher    2
Name: count, dtype: int64


##### reviews table

In [14]:
# Descriptive stats for quanititative variables in the reviews table.
reviews["review_score"].describe()

count    24934.000000
mean         4.096775
std          1.342658
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

In [15]:
# Create a frequency table for review score
reviews["review_score"].value_counts().sort_index()

review_score
1     2831
2      793
3     2019
4     4780
5    14511
Name: count, dtype: int64

##### products table

In [16]:
# Descriptive stats for quanititative variables in the products table.
# Product description lenght was purposly excluded.
products[["product_name_lenght", "product_photos_qty", "product_weight_g", 
          "product_length_cm", "product_height_cm", "product_width_cm"]].describe()

,product_name_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,13114.000000,13114.000000,13343.000000,13343.000000,13343.000000,13343.000000
mean,48.526537,2.218088,2192.937570,30.574758,16.788953,23.152140
std,10.238673,1.751910,4052.917466,16.616449,13.525831,12.097029
min,5.000000,1.000000,2.000000,7.000000,2.000000,7.000000
25%,42.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,3.000000,1850.000000,38.000000,20.000000,30.000000
max,76.000000,18.000000,30000.000000,105.000000,105.000000,118.000000


##### Descriptive Statistics Observations:
Across the profiled tables, several variables show a right-skew pattern, with means above the medians. price, freight_value, payment_installments, and product_weight_g. Payment_installments has a max of 24, a possible signal for cancellation risk given the financial commitment involved. Product_weight_g looks like the most heavily skewed variable, with a max of 30,000g compared with a median of just 700g, supporting the hypothesis that heavier or bulkier products may be more prone to shipping delays. Review_score is concentrated at 5 (14,511 of 24,934 reviews) but shows a meaningful grouping of very low scores (2,831 at score 1), giving a workable sample size for treating review_score as a problem-order signal; scores stayed correctly within the expected 1-5 range with no invalid values. Product_photos_qty also had no invalid values (minimum of 1, never 0), and its count (13,114) correctly excludes the 230 incomplete product listings identified earlier, which is consistent with prior findings. Two edge cases, a $0 payment fully covered by voucher, and 108 items with $0 freight. These were regular observation not errors.

### Frequency Tables (Categorical Variables)

In [17]:
# Frequency table for order status
orders["order_status"].value_counts()

order_status
delivered      24222
shipped          309
unavailable      167
canceled         154
invoiced          75
processing        71
approved           1
created            1
Name: count, dtype: int64

In [18]:
# Frequency table for payment type.
payments["payment_type"].value_counts()

payment_type
credit_card    19302
boleto          4973
voucher         1488
debit_card       391
Name: count, dtype: int64

In [19]:
# Frequency table for product category name.
product_category_names = products["product_category_name"].value_counts()
product_category_names.head(10)

product_category_name
cama_mesa_banho           1266
esporte_lazer             1167
moveis_decoracao          1022
utilidades_domesticas      978
beleza_saude               977
informatica_acessorios     700
automotivo                 679
relogios_presentes         565
brinquedos                 561
telefonia                  517
Name: count, dtype: int64

In [20]:
# Check the number of unique product category names.
products["product_category_name"].nunique()

72

In [21]:
# Check if all product category names in the products table are present in the translations table.
# Step 1: Create sets of unique product category names from both tables.
product_categories = set(products["product_category_name"].dropna().unique())
translation_categories = set(translations["product_category_name"].unique())

# Step 2: Compare the two sets to find any product category names in the products table that are not present in the translations table.
print(product_categories - translation_categories)

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


In [22]:
# Frequency table for customer state
customers["customer_state"].value_counts()

customer_state
SP    10478
RJ     3313
MG     2933
RS     1403
PR     1262
SC      860
BA      851
ES      524
DF      505
GO      498
PE      417
CE      329
PA      221
MT      212
MS      182
MA      160
PB      147
PI      130
RN      120
AL      107
SE      100
TO       74
RO       72
AM       42
AC       25
AP       20
RR       15
Name: count, dtype: int64

In [23]:
# Frequency table for seller state
sellers["seller_state"].value_counts()

seller_state
SP    1362
PR     247
MG     170
SC     136
RJ     119
RS      84
DF      27
GO      26
ES      16
BA      13
CE       7
PE       6
MS       4
RN       4
PB       4
MT       3
RO       2
PI       1
SE       1
MA       1
PA       1
Name: count, dtype: int64

##### Frequency Table Observations:
* Order status shows that cancellation is a rare occurrence, showing that of 25,000 orders, 24,222 (96.9%) were delivered and only 154 (0.6%) were canceled. This imbalance will matter for later modeling, since a model could be highly accurate by predicting "not canceled" every time, without being useful for identifying at-risk orders. Handling this will likely involve techniques such as resampling, adjusting how misclassifications are weighted, or shifting toward the broader "problem order" definition discussed, which would produce a larger and more workable positive class than cancellation by itself. The remaining 624 orders fall into other non-delivered statuses (shipped, unavailable, invoiced, processing, approved, created), some of them may also belong in that broader problem-order business problem definition.

* Payment type is concentrated in credit_card (19,302 of 26,154, about 74%), followed by boleto (4,973), voucher (1,488), and debit_card (391). 

* Product category name revealed a discrepancy worth flagging. The products table contains 72 unique categories, while the translations table contains only 71. Comparing the two directly identified two categories that are in products with no corresponding translation row ("pc_gamer" and "portateis_cozinha_e_preparadores_de_alimentos"), which will need to be addressed in the cleaning plan.

* Customer and seller state distributions both show heavy concentration in São Paulo (SP), but sellers are more concentrated than customers. SP accounts for about 61% of sellers (1,362 of 2,234) versus 42% of customers (10,478 of 25,000). This gap supports the hypothesis that a large share of orders are fulfilled across long distances, since customers outside the surrounding area might be buying from distant SP-based sellers. This could be tested more directly using the geolocation table to estimate seller-to-customer distance.

In [24]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


orders


,dtype
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


,missing_rate
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
order_approved_at,0.00156
order_id,0.00000
customer_id,0.00000
order_status,0.00000
order_purchase_timestamp,0.00000
order_estimated_delivery_date,0.00000



customers


,dtype
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


,missing_rate
customer_id,0.0
customer_unique_id,0.0
customer_zip_code_prefix,0.0
customer_city,0.0
customer_state,0.0


... (17 outputs left out)

,missing_rate
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0
geolocation_records,0.0



translations


,dtype
product_category_name,object
product_category_name_english,object


,missing_rate
product_category_name,0.0
product_category_name_english,0.0


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what key defines one row, and which tables were joined or aggregated.

Critical integration reminders:

- Enrich item records with product, category, and seller attributes while they are still item-level.
- Aggregate item, payment, and review records separately to `order_id`.
- Join each order-level summary to `orders` with merge validation.
- Treat the order-level ABT as trustworthy only after its row count and unique key have been checked.

The starter below demonstrates this sequence. Revise the variables to fit your question, and explain why each retained variable is useful.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [25]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.

# Check the data types of the timestamp columns 
time_stamp_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
            "order_delivered_customer_date", "order_estimated_delivery_date"]
print(orders[time_stamp_cols].dtypes)

# Check the data types of the zip code prefix columns
print(customers["customer_zip_code_prefix"].dtype, geo["geolocation_zip_code_prefix"].dtype)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
int64 int64


In [26]:

# Check for suspicious values in the items, payments, and products tables
print("price <= 0:", (items["price"] <= 0).sum())
print("freight < 0:", (items["freight_value"] < 0).sum())
print("payment_value == 0:", (payments["payment_value"] == 0).sum())
print("weight == 0:", (products["product_weight_g"] == 0).sum())

price <= 0: 0
freight < 0: 0
payment_value == 0: 2
weight == 0: 0


In [27]:
products_enriched = products.merge(
    translations,
    on="product_category_name",
    how="left",
    validate="many_to_one",
)

item_detail = (
    items
    .merge(
        products_enriched[[
            "product_id", "product_category_name_english", "product_weight_g"
        ]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        sellers[["seller_id", "seller_state"]],
        on="seller_id",
        how="left",
        validate="many_to_one",
    )
)

item_summary = (
    item_detail.groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_name_english", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        product_weight_mean_g=("product_weight_g", "mean"),
    )
    .reset_index()
)

payment_summary = (
    payments.groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
    )
    .reset_index()
)

reviews_for_summary = reviews.assign(
    review_comment_present=reviews["review_comment_message"].notna().astype(int)
)
review_summary = (
    reviews_for_summary.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        review_score_mean=("review_score", "mean"),
        review_comment_present=("review_comment_present", "max"),
    )
    .reset_index()
)

customer_geo = customers.merge(
    geo[[
        "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
        "geolocation_records"
    ]],
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng",
    "geolocation_records": "customer_geo_source_records",
})

abt = (
    orders
    .merge(customer_geo, on="customer_id", how="left", validate="one_to_one")
    .merge(item_summary, on="order_id", how="left", validate="one_to_one")
    .merge(payment_summary, on="order_id", how="left", validate="one_to_one")
    .merge(review_summary, on="order_id", how="left", validate="one_to_one")
)

abt["purchase_month"] = abt["order_purchase_timestamp"].dt.month
abt["purchase_dayofweek"] = abt["order_purchase_timestamp"].dt.dayofweek
abt["promised_delivery_days"] = (
    abt["order_estimated_delivery_date"] - abt["order_purchase_timestamp"]
).dt.days
abt["delivery_days"] = (
    abt["order_delivered_customer_date"] - abt["order_purchase_timestamp"]
).dt.days

delivery_outcome_observed = (
    abt["order_status"].eq("delivered")
    & abt["order_delivered_customer_date"].notna()
    & abt["order_estimated_delivery_date"].notna()
)
abt["delivered_late"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
abt.loc[delivery_outcome_observed, "delivered_late"] = (
    abt.loc[delivery_outcome_observed, "order_delivered_customer_date"]
    > abt.loc[delivery_outcome_observed, "order_estimated_delivery_date"]
).astype(int)

abt["high_review_score"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
review_outcome_observed = abt["review_score_mean"].notna()
abt.loc[review_outcome_observed, "high_review_score"] = (
    abt.loc[review_outcome_observed, "review_score_mean"] >= 4
).astype(int)


In [28]:
abt_validation = pd.Series({
    "orders_before_merges": len(orders),
    "abt_rows_after_merges": len(abt),
    "unique_order_ids": abt["order_id"].nunique(),
    "duplicate_order_ids": abt["order_id"].duplicated().sum(),
    "orders_with_item_summary": abt["order_item_count"].notna().sum(),
    "orders_with_payment_summary": abt["payment_total"].notna().sum(),
    "orders_with_review_summary": abt["review_score_mean"].notna().sum(),
    "orders_with_observed_delivery_target": abt["delivered_late"].notna().sum(),
    "orders_with_observed_review_target": abt["high_review_score"].notna().sum(),
})

display(abt_validation.to_frame("value"))
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


,value
orders_before_merges,25000
abt_rows_after_merges,25000
unique_order_ids,25000
duplicate_order_ids,0
orders_with_item_summary,24791
orders_with_payment_summary,25000
orders_with_review_summary,24797
orders_with_observed_delivery_target,24222
orders_with_observed_review_target,24797


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,customer_lat,customer_lng,customer_geo_source_records,order_item_count,product_count,seller_count,category_count,seller_state_count,item_price_total,freight_total,product_weight_mean_g,payment_record_count,payment_total,payment_installments_max,payment_type_count,review_record_count,review_score_mean,review_comment_present,purchase_month,purchase_dayofweek,promised_delivery_days,delivery_days,delivered_late,high_review_score
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,32ea3bdedab835c3aa6cb68ce66565ef,4106,sao paulo,SP,4106.0,-23.581451,-46.635029,45.0,1.0,1.0,1.0,1.0,1.0,29.90,15.56,300.0,1,45.46,1,1,1.0,4.0,1.0,10,0,23,23.0,0,1
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23,87776adb449c551e74c13fc34f036105,12030,taubate,SP,12030.0,-23.032142,-45.570461,218.0,1.0,1.0,1.0,1.0,1.0,29.99,10.96,9000.0,1,40.95,4,1,1.0,5.0,0.0,10,0,50,10.0,0,1
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24,823c47d4abda1f8ce7568145f76c2b85,13175,sumare,SP,13175.0,-22.817992,-47.241101,173.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,50.74,3,1,1.0,2.0,0.0,10,1,50,NaN,<NA>,0
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28,0829f7df6577d5a4b65439bea701405f,31210,belo horizonte,MG,31210.0,-19.906712,-43.948897,156.0,1.0,1.0,1.0,1.0,1.0,249.90,17.59,610.0,1,267.49,10,1,1.0,3.0,0.0,10,1,54,12.0,0,0
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28,df2988ba3ed226b10521a0e4da849b61,23092,rio de janeiro,RJ,23092.0,-22.870856,-43.543849,217.0,1.0,1.0,1.0,1.0,1.0,67.90,18.98,2100.0,7,86.88,1,1,1.0,5.0,1.0,10,1,54,51.0,0,1


,missing_rate
delivered_late,0.03112
delivery_days,0.03100
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
product_weight_mean_g,0.00844
seller_count,0.00836
order_item_count,0.00836
freight_total,0.00836
item_price_total,0.00836
seller_state_count,0.00836


In [29]:
# Create a copy of the abt dataframe to for cleaning and transformation 
olist_abt = abt.copy()

In [30]:
# Drop duplicate zip colunmn
olist_abt.drop(columns=["geolocation_zip_code_prefix"])

# Group order status
status_group_map = {
    "delivered": "delivered",
    "cancelled": "cancelled_or_unavailable",
    "unavailable": "cancelled_or_unavailable",
    "shipped": "unresolved", "invoiced": "unresolved", "processing": "unresolved",
    "approved": "unresolved", "created": "unresolved",
}
olist_abt["order_status_grouped"] = olist_abt["order_status"].map(status_group_map)

print(olist_abt["order_status_grouped"].value_counts(dropna=False))
print(olist_abt["order_status_grouped"].value_counts().sum())

order_status_grouped
delivered                   24222
unresolved                    457
cancelled_or_unavailable      167
NaN                           154
Name: count, dtype: int64
24846


In [31]:
# Get a column list of the ABT to verify the expected columns are present.
print(olist_abt.columns.tolist())
len(olist_abt)

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'geolocation_zip_code_prefix', 'customer_lat', 'customer_lng', 'customer_geo_source_records', 'order_item_count', 'product_count', 'seller_count', 'category_count', 'seller_state_count', 'item_price_total', 'freight_total', 'product_weight_mean_g', 'payment_record_count', 'payment_total', 'payment_installments_max', 'payment_type_count', 'review_record_count', 'review_score_mean', 'review_comment_present', 'purchase_month', 'purchase_dayofweek', 'promised_delivery_days', 'delivery_days', 'delivered_late', 'high_review_score', 'order_status_grouped']


25000

In [32]:
# Check shape and info
print(olist_abt.shape)
print(olist_abt.info())

(25000, 38)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 38 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       25000 non-null  object        
 1   customer_id                    25000 non-null  object        
 2   order_status                   25000 non-null  object        
 3   order_purchase_timestamp       25000 non-null  datetime64[ns]
 4   order_approved_at              24961 non-null  datetime64[ns]
 5   order_delivered_carrier_date   24548 non-null  datetime64[ns]
 6   order_delivered_customer_date  24225 non-null  datetime64[ns]
 7   order_estimated_delivery_date  25000 non-null  datetime64[ns]
 8   customer_unique_id             25000 non-null  object        
 9   customer_zip_code_prefix       25000 non-null  int64         
 10  customer_city                  25000 non-null  object        
 11  cus

In [33]:
# Check the types of the new data frame
print(olist_abt.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
customer_unique_id                       object
customer_zip_code_prefix                  int64
customer_city                            object
customer_state                           object
geolocation_zip_code_prefix             float64
customer_lat                            float64
customer_lng                            float64
customer_geo_source_records             float64
order_item_count                        float64
product_count                           float64
seller_count                            float64
category_count                          float64
seller_state_count                      

In [34]:
# Check the distribution of order_status in the ABT 
status_counts = olist_abt["order_status"].value_counts()
status_pct = olist_abt["order_status"].value_counts(normalize=True).mul(100).round(2)
print(pd.concat([status_counts, status_pct], axis=1, keys=["orders", "pct"]))

              orders    pct
order_status               
delivered      24222  96.89
shipped          309   1.24
unavailable      167   0.67
canceled         154   0.62
invoiced          75   0.30
processing        71   0.28
approved           1   0.00
created            1   0.00


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the order-level ABT row count, key uniqueness, and target eligibility without creating another permanent database file.


In [35]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("olist_order_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            SUM(CASE WHEN delivered_late IS NOT NULL THEN 1 ELSE 0 END) AS delivery_target_rows,
            SUM(CASE WHEN high_review_score IS NOT NULL THEN 1 ELSE 0 END) AS review_target_rows
        FROM olist_order_abt
        ''',
        connection,
    )

display(sql_abt_check)


,abt_rows,unique_order_ids,delivery_target_rows,review_target_rows
0,25000,25000,24222,24797


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [36]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,order_id,object,,,,
1,customer_id,object,,,,
2,order_status,object,,,,
3,order_purchase_timestamp,datetime64[ns],,,,
4,order_approved_at,datetime64[ns],,,,


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminders:

- If you model `delivered_late`, use only rows where the nullable target is defined; undelivered orders do not reveal a normal late/on-time outcome.
- For a prediction made at purchase time, do not use actual delivery duration, final order status, review fields, or other post-purchase outcomes as predictors.
- If you model `high_review_score`, use only rows with an observed review and do not use review-derived fields as predictors.
- Check class balance, keep a reproducible train/test split, and interpret precision and recall when the positive class is uncommon.
- Explain the intended decision and prediction timing before choosing features.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [37]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [38]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,25000,25000,3b697a20d9e427646d92567910af6d57,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,25000,25000,355077684019f7f60a031656bd7262b8,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,25000,8,delivered,24222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,25000,NaN,NaN,NaN,2017-12-31 22:18:29.561480192,2016-10-03 09:44:50,2017-09-13 11:52:50,2018-01-19 19:08:12.500000,2018-05-05 11:17:57,2018-10-01 15:30:09,NaN
order_approved_at,24961,NaN,NaN,NaN,2018-01-01 06:46:13.413765376,2016-10-04 10:19:23,2017-09-13 19:55:11,2018-01-20 09:08:59,2018-05-05 11:50:40,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,24548,NaN,NaN,NaN,2018-01-05 06:50:25.146529280,2016-10-08 10:34:01,2017-09-18 18:01:51.750000128,2018-01-24 20:01:44.500000,2018-05-08 14:21:00,2018-09-04 15:25:00,NaN
order_delivered_customer_date,24225,NaN,NaN,NaN,2018-01-14 19:15:33.384024832,2016-10-11 14:46:49,2017-09-26 19:04:55,2018-02-03 03:49:26,2018-05-16 12:26:50,2018-09-27 02:24:33,NaN
order_estimated_delivery_date,25000,NaN,NaN,NaN,2018-01-24 17:19:26.975999744,2016-10-27 00:00:00,2017-10-04 00:00:00,2018-02-15 00:00:00,2018-05-28 00:00:00,2018-10-23 00:00:00,NaN
customer_unique_id,25000,24764,8d50f5eadf50201ccdcedfb9e2ac8455,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,25000.0,NaN,NaN,NaN,34966.35068,1004.0,11431.75,24326.0,58037.0,99970.0,29706.603087


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [39]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "olist_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "olist_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "olist_model_metrics.csv"
# report_path = REPORTS_DIR / "olist_final_report.html"


In [40]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
